# CIGALE Decomposition Recombination Validation

Tests whether the paper's **theoretical** additive-mixing model
(`glass.composite_math.create_composite_sed`, used throughout the Part 1
synthetic SKIRTOR grid) is a valid stand-in for real, CIGALE-fit AGN-host
galaxies (Part 2).

**Methodology:** for every ZFOURGE galaxy with a non-zero CIGALE best-fit AGN
fraction (`fracAGN`), take the CIGALE-decomposed **host-only** SED
(`glass.analysis.decompose_cigale_sed`, i.e. the real photometry with
CIGALE's own best-fit AGN component subtracted) and add back an
**independent theoretical SKIRTOR AGN template** (the paper's fixed Type 1 /
face-on and Type 2 / edge-on templates, `config.SKIRTOR_TYPE1_PARAMS` /
`SKIRTOR_TYPE2_PARAMS`) via the paper's own `alpha`-mixing formalism, with
`alpha` predicted *a priori* from `fracAGN`
(`alpha = fracAGN / (1 - fracAGN)`, from the paper's own definition that the
AGN's share of total integrated flux is `alpha/(1+alpha)`). The
reconstruction is then compared against the real total (CIGALE
`L_lambda_total`) SED and against the real `UV_Full`/`VJ_Full` colours
already tabulated for the population.

This is a genuine test, **not** a self-consistency check: unlike summing
CIGALE's own internal `agn.SKIRTOR2016_*` components back onto its own host
split (which reproduces the total exactly, by construction), reconstructing
with an independent theoretical template can - and, as shown below,
sometimes does - disagree with reality. Supersedes the single-galaxy demo in
`scripts/recreate_theoretical_results.py::recreate_cigale_decomposition_demo`.

Population: all `fracAGN > 0` galaxies (n~6509), matching the sample used in
`docs/figure9_10_bootstrap_methodology.md`.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.append(os.path.abspath('..'))

from src import config
from glass import data_io, composite_math, photometry, visualization, analysis

plt.style.use('default')
visualization.apply_pasa_style()
os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)

VALIDATION_OUTPUT_DIR = os.path.join(config.PROCESSED_DATA_DIR, 'cigale_theoretical_validation')
os.makedirs(VALIDATION_OUTPUT_DIR, exist_ok=True)

## 1. Population, theoretical AGN templates, and caching

Loads the same `fracAGN > 0` population used in `CIGALE_Decomposition_Analysis.ipynb`,
the paper's fixed Type 1/Type 2 SKIRTOR templates (read once, not per galaxy), and the
U/V/J passbands. The per-galaxy reconstruction loop below is cached to disk
(`outputs/cigale_theoretical_validation/`) since it touches ~6500 individual CIGALE
FITS files and takes several minutes on first run.

In [ ]:
cigale_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'zfourge_full_final.csv')
agn_frac_csv = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed', 'agn_fractions.csv')

df_cig = pd.read_csv(cigale_csv, low_memory=False)
z_col = 'zpk_x' if 'zpk_x' in df_cig.columns else 'zpk'

# Reuses the exact fracAGN-caching pattern from CIGALE_Decomposition_Analysis.ipynb
# (cell 6): build agn_fractions.csv from the FITS headers if it doesn't exist yet.
if not os.path.exists(agn_frac_csv):
    from astropy.io import fits as _fits
    _rows = []
    for _gid, _field in zip(df_cig['ID'], df_cig['field']):
        _path = os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed',
                              f"{_field.lower()}_best_models_fits",
                              f"{_gid.split('_', 1)[1]}_best_model.fits")
        if os.path.exists(_path):
            _rows.append({'ID': _gid, 'fracAGN': float(_fits.getheader(_path, 1)['agn.fracAGN'])})
    pd.DataFrame(_rows).to_csv(agn_frac_csv, index=False)

df_cig = df_cig.merge(pd.read_csv(agn_frac_csv), on='ID', how='left')

AGN_FRAC_MIN = 0.0
df_agn = df_cig[df_cig['fracAGN'] > AGN_FRAC_MIN].reset_index(drop=True)
print(f"AGN-hosting galaxies (fracAGN > {AGN_FRAC_MIN}): {len(df_agn)} of {len(df_cig)}")

In [ ]:
skirtor_dir = os.path.join(config.RAW_DATA_DIR, 'Templates', 'Skirtor')
agn_type1 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE1_PARAMS)
agn_type2 = data_io.read_skirtor_model(skirtor_dir, **config.SKIRTOR_TYPE2_PARAMS)
TEMPLATES = {'Type1': agn_type1, 'Type2': agn_type2}
TEMPLATE_COLORS = {'Type1': '#1A6FB5', 'Type2': '#CC2929'}

filters = photometry.load_passbands(config.FILTER_PATHS)

print(f"Type1 (face-on, i={config.SKIRTOR_TYPE1_PARAMS['inclination']}) wavelength range: "
      f"{agn_type1['lambda (Angstroms)'].min():.1f}-{agn_type1['lambda (Angstroms)'].max():.2e} A")
print(f"Type2 (edge-on, i={config.SKIRTOR_TYPE2_PARAMS['inclination']}) wavelength range: "
      f"{agn_type2['lambda (Angstroms)'].min():.1f}-{agn_type2['lambda (Angstroms)'].max():.2e} A")

**A note on column reuse (read this before the reconstruction loop):**
`data_io.read_cigale_best_model()` writes a `Fnu`-derived
`'Total Flux (erg/s/cm^2/Angstrom)'` column; `analysis.decompose_cigale_sed(...,
target='host')` then **overwrites** that same column with an
`L_lambda_total`-derived value. Calling them in that order (as below) is
deliberate and safe - it's how the rest of this notebook, and the legacy
pipeline that produced the `UV_Full`/`VJ_Full` columns in
`zfourge_full_final.csv`, both derive "flux" from CIGALE's own rest-frame
luminosity density rather than the observed-frame `Fnu` column. The
ground-truth "full" SED used for every residual below is read from
`L_lambda_total` directly, before decomposition - never from the reused
`'Total Flux...'` column name.

Relative flux residuals (Tier 1 only - the colour comparisons in Tier 2 are
unaffected) are only evaluated where the true flux exceeds `1e-4` of that
galaxy's peak `L_lambda_total`. Below that floor, CIGALE's own model flux is
numerically negligible and a fractional residual there is dominated by
division-by-near-zero rather than anything physical.

The population-level wavelength figure below (Figure 1, Panel A) shows the
*full* wavelength range including a genuine, large blow-up shortward of
~1300 A restframe: CIGALE's `igm` column applies Lyman-continuum/IGM
absorption to the true total SED there, which this notebook's naive additive
reconstruction does not replicate (it has no absorption physics at all) - a
real, expected limitation of the theoretical model in a regime that plays no
role in the U/V/J passbands used everywhere else in this notebook (U's
restframe effective wavelength is ~3600 A). Per-galaxy summary statistics
(Panel B's histogram, and the Section 6 pass/fail check) are therefore
restricted to `wavelength > 1300` A, so a single far-UV/EUV outlier bin
doesn't dominate a metric meant to describe reconstruction quality in the
astrophysically relevant range.

In [ ]:
FLUX_FLOOR_FRACTION = 1e-4
WAVELENGTH_VALID_MIN = 1300.0  # Angstroms restframe; excludes the Lyman-continuum/IGM regime
WL_BIN_EDGES = np.logspace(1, 7, 41)  # 10 A to 1e7 A (SKIRTOR's own native range), 40 bins
WL_BIN_CENTERS = np.sqrt(WL_BIN_EDGES[:-1] * WL_BIN_EDGES[1:])


def _fits_path(gid, field):
    fits_num = gid.split('_', 1)[1]
    return os.path.join(config.RAW_DATA_DIR, 'full_zfourge_decomposed',
                         f"{field.lower()}_best_models_fits", f"{fits_num}_best_model.fits")


def process_galaxy(gid, field, z, fracAGN):
    """
    Reconstructs galaxy `gid`'s full SED as host (CIGALE-decomposed) +
    alpha_theory * SKIRTOR template, for both Type1 and Type2, and returns a
    summary row plus per-template per-wavelength-bin median residuals (for
    the population-level wavelength figure).
    """
    path = _fits_path(gid, field)
    if not os.path.exists(path):
        return None, None

    full_sed = data_io.read_cigale_best_model(path, redshift=z, restframe=True)
    wl_full = full_sed['lambda (Angstroms)'].values.astype(float)
    full_L = full_sed['L_lambda_total'].values.astype(float)
    peak_L = np.nanmax(full_L)
    floor = FLUX_FLOOR_FRACTION * peak_L if peak_L > 0 else 0.0

    n_agn_components_present = sum(
        c in full_sed.columns for c in
        ['agn.SKIRTOR2016_torus', 'agn.SKIRTOR2016_polar_dust', 'agn.SKIRTOR2016_disk']
    )

    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')
    alpha_theory = fracAGN / (1.0 - fracAGN) if fracAGN < 1.0 else np.nan

    row = {
        'ID': gid, 'field': field, 'fracAGN': fracAGN, 'redshift': z,
        'n_agn_components_present': n_agn_components_present, 'alpha_theory': alpha_theory,
    }
    binned = {}

    for tname, agn_template in TEMPLATES.items():
        composite = composite_math.create_composite_sed(agn_template, host_sed, alpha_theory)
        wl_c = composite['lambda (Angstroms)'].values
        flux_c = composite['Total Flux (erg/s/cm^2/Angstrom)'].values

        full_interp = np.interp(wl_c, wl_full, full_L, left=np.nan, right=np.nan)
        valid = np.isfinite(full_interp) & (full_interp > floor)
        resid = np.full_like(flux_c, np.nan)
        resid[valid] = (flux_c[valid] - full_interp[valid]) / full_interp[valid]

        # Per-galaxy summary stats are restricted to the astrophysically relevant
        # range (excludes the Lyman-continuum/IGM regime - see the markdown note
        # above); the wavelength-binned figure below uses the unrestricted `valid`
        # mask so that blow-up is visible rather than hidden.
        valid_summary = valid & (wl_c > WAVELENGTH_VALID_MIN)
        resid_summary = resid[valid_summary]
        row[f'resid_median_{tname}'] = np.median(resid_summary) if valid_summary.any() else np.nan
        row[f'max_abs_resid_{tname}'] = np.max(np.abs(resid_summary)) if valid_summary.any() else np.nan

        bin_idx = np.digitize(wl_c[valid], WL_BIN_EDGES) - 1
        bin_medians = np.full(len(WL_BIN_CENTERS), np.nan)
        r_valid = resid[valid]
        in_range = (bin_idx >= 0) & (bin_idx < len(WL_BIN_CENTERS))
        for b in np.unique(bin_idx[in_range]):
            bin_medians[b] = np.median(r_valid[bin_idx == b])
        binned[tname] = bin_medians

        # Colour computation can hit a math-domain error (log10 of a non-positive
        # flux ratio) when the composite's flux goes negative or zero within a
        # passband - the theoretical template's negative excursions relative to
        # the host aren't always physical for a real galaxy. Treated as a genuine
        # reconstruction failure (tracked via cls_recombined=-1) rather than
        # silently discarded, so its rate is reported alongside residual stats.
        try:
            uv_r, vj_r = photometry.calculate_UVJ_colours(composite, filters['U'], filters['V'], filters['J'])
            cls_r = int(photometry.classify_uvj(np.array([vj_r]), np.array([uv_r]))[0])
        except (ValueError, ZeroDivisionError):
            uv_r, vj_r, cls_r = np.nan, np.nan, -1
        row[f'UV_recombined_{tname}'] = uv_r
        row[f'VJ_recombined_{tname}'] = vj_r
        row[f'cls_recombined_{tname}'] = cls_r

        # Secondary diagnostic: closed-form best-fit alpha (least squares) using the
        # same aligned/scaled AGN template create_composite_sed itself computes.
        wl_al, agn_al, host_al = composite_math.adjust_wavelength_range(
            agn_template['lambda (Angstroms)'].values, agn_template['Total Flux (erg/s/cm^2/Angstrom)'].values,
            host_sed['lambda (Angstroms)'].values, host_sed['Total Flux (erg/s/cm^2/Angstrom)'].values)
        S = composite_math.compute_scaling_factor(wl_al, agn_al, wl_al, host_al)
        scaled_agn = agn_al * S
        full_al = np.interp(wl_al, wl_full, full_L, left=np.nan, right=np.nan)
        m = np.isfinite(full_al) & np.isfinite(host_al) & np.isfinite(scaled_agn) & (full_al > floor)
        denom = np.sum(scaled_agn[m] ** 2)
        row[f'alpha_fit_{tname}'] = (
            np.sum((full_al[m] - host_al[m]) * scaled_agn[m]) / denom if denom > 0 else np.nan
        )

    return row, binned

In [ ]:
SUMMARY_CSV = os.path.join(VALIDATION_OUTPUT_DIR, 'reconstruction_summary.csv')
WLBIN_CSV = os.path.join(VALIDATION_OUTPUT_DIR, 'reconstruction_residual_by_wavelength.csv')

if os.path.exists(SUMMARY_CSV) and os.path.exists(WLBIN_CSV):
    summary_df = pd.read_csv(SUMMARY_CSV)
    wlbin_df = pd.read_csv(WLBIN_CSV)
    print(f"Loaded cached results for {len(summary_df)} galaxies.")
else:
    rows = []
    binned_store = {'Type1': [], 'Type2': []}
    t_start = time.time()
    n_missing = 0
    for _, r in df_agn.iterrows():
        row, binned = process_galaxy(r['ID'], r['field'], r[z_col], r['fracAGN'])
        if row is None:
            n_missing += 1
            continue
        rows.append(row)
        for tname in TEMPLATES:
            binned_store[tname].append(binned[tname])
        if len(rows) % 500 == 0:
            elapsed = time.time() - t_start
            print(f"  {len(rows)}/{len(df_agn)} galaxies processed ({elapsed:.0f}s elapsed)")

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(SUMMARY_CSV, index=False)

    wlbin_rows = []
    for tname in TEMPLATES:
        arr = np.array(binned_store[tname])  # (n_galaxies, n_bins)
        med = np.nanmedian(arr, axis=0)
        p16 = np.nanpercentile(arr, 16, axis=0)
        p84 = np.nanpercentile(arr, 84, axis=0)
        n_gal_in_bin = np.sum(np.isfinite(arr), axis=0)
        for b in range(len(WL_BIN_CENTERS)):
            wlbin_rows.append({'template': tname, 'wavelength_A': WL_BIN_CENTERS[b],
                                'median_resid': med[b], 'p16_resid': p16[b], 'p84_resid': p84[b],
                                'n_galaxies': int(n_gal_in_bin[b])})
    wlbin_df = pd.DataFrame(wlbin_rows)
    wlbin_df.to_csv(WLBIN_CSV, index=False)

    print(f"Done: {len(summary_df)} galaxies processed, {n_missing} missing FITS files skipped, "
          f"{time.time() - t_start:.0f}s total.")

## 2. Tier 1 - Flux-level reconstruction fidelity (Figure 1)

Unlike re-summing CIGALE's own internal `agn.SKIRTOR2016_*` components (which
reproduces the total exactly, by construction), the reconstruction here uses
an *independent* theoretical AGN template, so it can genuinely disagree with
the real SED. This section checks how well it agrees, across the full
population and across wavelength - a fast screen before the more
scientifically decisive colour-level test in Section 3.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=visualization.PASA_WIDE)

ax = axes[0]
for tname in TEMPLATES:
    sub = wlbin_df[wlbin_df['template'] == tname].sort_values('wavelength_A')
    ax.plot(sub['wavelength_A'], sub['median_resid'], color=TEMPLATE_COLORS[tname], label=tname, lw=1.5)
    ax.fill_between(sub['wavelength_A'], sub['p16_resid'], sub['p84_resid'], color=TEMPLATE_COLORS[tname], alpha=0.2)
ax.axhline(0, color='k', lw=0.8, ls=':')
ax.axvline(WAVELENGTH_VALID_MIN, color='gray', lw=1, ls='--', label='1300 A (Lyman-cont. cut)')
ax.set_xscale('log')
ax.set_xlabel('Restframe wavelength (A)')
ax.set_ylabel('(Composite - Full) / Full')
ax.set_title('Population median residual vs wavelength (full range)')
ax.legend(fontsize=7)

ax = axes[1]
for tname in TEMPLATES:
    vals = summary_df[f'max_abs_resid_{tname}'].replace([np.inf, -np.inf], np.nan).dropna()
    vals = vals[vals > 0]
    ax.hist(np.log10(vals), bins=50, color=TEMPLATE_COLORS[tname], alpha=0.5, label=tname, density=True)
ax.set_xlabel('log10(per-galaxy max |relative residual|), wavelength > 1300 A')
ax.set_ylabel('Density')
ax.set_title('Per-galaxy worst-case residual (astrophysically relevant range)')
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(VALIDATION_OUTPUT_DIR, 'Figure1_flux_reconstruction_fidelity.png'), dpi=300, bbox_inches='tight')
plt.show()

for tname in TEMPLATES:
    med = summary_df[f'resid_median_{tname}']
    mx = summary_df[f'max_abs_resid_{tname}'].dropna()
    print(f"{tname} (wavelength > 1300 A): population median residual = {np.nanmedian(med):+.4f}, "
          f"95th pct per-galaxy max |residual| = {np.nanpercentile(mx, 95):.3f}")
print(f"Galaxies with n_agn_components_present < 3: {(summary_df['n_agn_components_present'] < 3).sum()}")

OUTLIER_THRESHOLD = 1.0  # 100% relative residual
for tname in TEMPLATES:
    bad = summary_df[summary_df[f'max_abs_resid_{tname}'] > OUTLIER_THRESHOLD].sort_values(
        f'max_abs_resid_{tname}', ascending=False)
    print(f"{tname}: {len(bad)} galaxies with max |residual| > {OUTLIER_THRESHOLD:.0%}; "
          f"worst 10: {list(bad['ID'].head(10))}")

## 3. Tier 2 - UVJ colour-level validation (Figure 2)

The real test: `UV_Full`/`VJ_Full` in `zfourge_full_final.csv` were computed
by an independent (legacy) pipeline from CIGALE's actual total SED. If the
theoretical reconstruction's colours match these, the paper's Part 1 grid is
a trustworthy stand-in for real galaxies; if not, the gap is itself a
finding.

In [ ]:
def bootstrap_ci(values, stat_fn=np.median, n_boot=2000, seed=42, ci=(2.5, 97.5)):
    """
    Non-parametric percentile bootstrap CI, following the convention in
    docs/figure9_10_bootstrap_methodology.md: resample `values` with
    replacement (same size as input), apply stat_fn along axis=1 across
    n_boot resamples, return (point_estimate, lo, hi).
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    point = stat_fn(values)
    rng = np.random.default_rng(seed)
    boot = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_stat = stat_fn(boot, axis=1)
    lo, hi = np.percentile(boot_stat, ci)
    return point, lo, hi


catalog_cols = summary_df.merge(df_cig[['ID', 'UV_Full', 'VJ_Full']], on='ID', how='left')
catalog_cols['cls_full'] = photometry.classify_uvj(catalog_cols['VJ_Full'], catalog_cols['UV_Full'])
for tname in TEMPLATES:
    catalog_cols[f'dUV_{tname}'] = catalog_cols[f'UV_recombined_{tname}'] - catalog_cols['UV_Full']
    catalog_cols[f'dVJ_{tname}'] = catalog_cols[f'VJ_recombined_{tname}'] - catalog_cols['VJ_Full']

COLOUR_TOLERANCE = 0.02  # mag; small relative to the UVJ region-boundary geometry in photometry.classify_uvj

for i_t, tname in enumerate(TEMPLATES):
    duv = catalog_cols[f'dUV_{tname}']
    dvj = catalog_cols[f'dVJ_{tname}']
    m_uv, lo_uv, hi_uv = bootstrap_ci(np.abs(duv), np.median, seed=201 + i_t)
    m_vj, lo_vj, hi_vj = bootstrap_ci(np.abs(dvj), np.median, seed=211 + i_t)
    frac_above = ((np.abs(duv) > COLOUR_TOLERANCE) | (np.abs(dvj) > COLOUR_TOLERANCE)).mean()
    n_fail = (catalog_cols[f'cls_recombined_{tname}'] == -1).sum()
    print(f"{tname}: median|dUV|={m_uv:.4f} [{lo_uv:.4f},{hi_uv:.4f}], "
          f"median|dVJ|={m_vj:.4f} [{lo_vj:.4f},{hi_vj:.4f}], "
          f"{frac_above:.1%} of galaxies exceed {COLOUR_TOLERANCE} mag in UV or VJ, "
          f"{n_fail} ({n_fail / len(catalog_cols):.2%}) had a non-physical (negative/zero flux) "
          f"colour-computation failure and are excluded from these stats")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(visualization.PASA_WIDE[0] * 1.4, visualization.PASA_WIDE[1]))

ax = axes[0]
for tname in TEMPLATES:
    ax.scatter(catalog_cols[f'dVJ_{tname}'], catalog_cols[f'dUV_{tname}'], s=3, alpha=0.15,
               color=TEMPLATE_COLORS[tname], label=tname)
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set_xlabel('dVJ = VJ_recombined - VJ_Full')
ax.set_ylabel('dUV = UV_recombined - UV_Full')
ax.set_title('Colour residuals')
ax.legend(fontsize=7, markerscale=3)

ax = axes[1]
for tname in TEMPLATES:
    combined = pd.concat([catalog_cols[f'dUV_{tname}'].abs(), catalog_cols[f'dVJ_{tname}'].abs()])
    ax.hist(combined.clip(upper=0.5), bins=60, histtype='step', color=TEMPLATE_COLORS[tname], label=tname, density=True)
ax.axvline(COLOUR_TOLERANCE, color='gray', lw=1, ls='--', label=f'{COLOUR_TOLERANCE} mag tol.')
ax.set_xlabel('|dUV| or |dVJ| (mag)')
ax.set_ylabel('Density')
ax.set_title('Residual magnitude')
ax.legend(fontsize=7)

ax = axes[2]
region_names = {0: 'Quiescent', 1: 'Star-forming', 2: 'Dusty'}
x = np.arange(3)
width = 0.35
for i_t, tname in enumerate(TEMPLATES):
    agree = []
    for cid in [0, 1, 2]:
        mask = (catalog_cols['cls_full'] == cid) & (catalog_cols[f'cls_recombined_{tname}'] != -1)
        agree.append((catalog_cols.loc[mask, f'cls_recombined_{tname}'] == cid).mean() if mask.sum() else np.nan)
    ax.bar(x + (i_t - 0.5) * width, agree, width, color=TEMPLATE_COLORS[tname], label=tname)
ax.set_xticks(x)
ax.set_xticklabels([region_names[c] for c in [0, 1, 2]])
ax.set_ylabel('Classification agreement (reconstructed vs Full)')
ax.set_ylim(0, 1.05)
ax.set_title('UVJ classification agreement')
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(VALIDATION_OUTPUT_DIR, 'Figure2_UVJ_colour_validation.png'), dpi=300, bbox_inches='tight')
plt.show()

overall_agreement = {}
for tname in TEMPLATES:
    ok = catalog_cols[f'cls_recombined_{tname}'] != -1
    overall_agreement[tname] = (
        catalog_cols.loc[ok, 'cls_full'] == catalog_cols.loc[ok, f'cls_recombined_{tname}']
    ).mean()
print("Overall UVJ classification agreement (reconstructed vs true Full, excluding colour-computation failures):",
      overall_agreement)

## 4. Alpha calibration diagnostic (Figure 3)

Secondary check, not the headline result: does the a priori
`alpha_theory = fracAGN / (1 - fracAGN)` match the per-galaxy best-fit
`alpha_fit` (closed-form least squares against the true full SED)? A
systematic offset here would point to a mismatch between CIGALE's `fracAGN`
definition (conventionally an IR/dust-luminosity fraction) and the paper's
alpha (a whole-SED integrated-flux fraction), rather than a failure of the
additive-mixing formalism itself.

In [ ]:
best_template = min(
    TEMPLATES,
    key=lambda t: np.nanmedian(catalog_cols[f'dUV_{t}'].abs() + catalog_cols[f'dVJ_{t}'].abs())
)
print(f"Using {best_template} (better Tier-2 colour match) for the alpha calibration check.")

fig, ax = plt.subplots(figsize=visualization.PASA_SQ)
at = catalog_cols['alpha_theory']
af = catalog_cols[f'alpha_fit_{best_template}']
valid = np.isfinite(at) & np.isfinite(af) & (at > 0) & (af > 0)
ax.scatter(at[valid], af[valid], s=3, alpha=0.15, color=TEMPLATE_COLORS[best_template])
lims = [min(at[valid].min(), af[valid].min()), max(at[valid].max(), af[valid].max())]
ax.plot(lims, lims, 'k--', lw=1, label='1:1')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('alpha_theory = fracAGN / (1 - fracAGN)')
ax.set_ylabel(f'alpha_fit ({best_template}, best-fit least squares)')
ax.legend(fontsize=7)

rho, pval = stats.spearmanr(at[valid], af[valid])
ratio = af[valid] / at[valid]
m_ratio, lo_ratio, hi_ratio = bootstrap_ci(ratio, np.median, seed=301)
ax.set_title(f"Spearman rho={rho:.3f}, median ratio={m_ratio:.2f} [{lo_ratio:.2f},{hi_ratio:.2f}]", fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(VALIDATION_OUTPUT_DIR, 'Figure3_alpha_calibration.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"n valid = {valid.sum()}, Spearman rho={rho:.3f} (p={pval:.2e}), "
      f"median alpha_fit/alpha_theory = {m_ratio:.3f} [{lo_ratio:.3f}, {hi_ratio:.3f}]")

## 5. Residual stability vs fracAGN and redshift (Figure 4)

Checks whether the theoretical reconstruction's colour fidelity degrades in
exactly the small-N, high-`fracAGN`/high-z tail where the population-level
Fig. 9/10 results (`docs/figure9_10_bootstrap_methodology.md`) are already
known to be statistically fragile.

In [ ]:
catalog_cols[f'vecmag_{best_template}'] = analysis.calculate_vector_magnitude(
    catalog_cols[f'VJ_recombined_{best_template}'], catalog_cols[f'UV_recombined_{best_template}'],
    catalog_cols['VJ_Full'], catalog_cols['UV_Full'])

frac_grid = sorted(catalog_cols['fracAGN'].unique())
z_bins = [(0, 0.5), (0.5, 1.0), (1.0, 1.5), (1.5, 2.0), (2.0, 3.5)]

fig, axes = plt.subplots(1, 2, figsize=visualization.PASA_WIDE)

ax = axes[0]
pts, los, his = [], [], []
for i_f, fv in enumerate(frac_grid):
    vals = catalog_cols.loc[catalog_cols['fracAGN'] == fv, f'vecmag_{best_template}']
    p, lo, hi = bootstrap_ci(vals, np.median, seed=400 + i_f)
    pts.append(p); los.append(lo); his.append(hi)
pts, los, his = np.array(pts), np.array(los), np.array(his)
ax.errorbar(frac_grid, pts, yerr=[pts - los, his - pts], fmt='o-', color=TEMPLATE_COLORS[best_template], capsize=2)
ax.set_xlabel('CIGALE fracAGN')
ax.set_ylabel('Median UVJ residual magnitude')
ax.set_title(f'{best_template}: residual vs fracAGN')

ax = axes[1]
pts, los, his, centers = [], [], [], []
for i_z, (zlo, zhi) in enumerate(z_bins):
    vals = catalog_cols.loc[(catalog_cols['redshift'] >= zlo) & (catalog_cols['redshift'] < zhi),
                             f'vecmag_{best_template}']
    p, lo, hi = bootstrap_ci(vals, np.median, seed=500 + i_z)
    pts.append(p); los.append(lo); his.append(hi); centers.append((zlo + zhi) / 2)
pts, los, his = np.array(pts), np.array(los), np.array(his)
ax.errorbar(centers, pts, yerr=[pts - los, his - pts], fmt='o-', color=TEMPLATE_COLORS[best_template], capsize=2)
ax.set_xlabel('Redshift')
ax.set_ylabel('Median UVJ residual magnitude')
ax.set_title(f'{best_template}: residual vs redshift')

plt.tight_layout()
fig.savefig(os.path.join(VALIDATION_OUTPUT_DIR, 'Figure4_residual_stability.png'), dpi=300, bbox_inches='tight')
plt.show()

## 6. Verification: single-galaxy spot-check and pass/fail summary

Re-runs the same CDFS/5886 galaxy used in
`scripts/recreate_theoretical_results.py`'s single-galaxy demo through this
notebook's full pipeline, for a direct visual/numeric cross-check.

In [ ]:
spot_id = 'CDFS_5886'
spot_row = catalog_cols[catalog_cols['ID'] == spot_id]
if len(spot_row) == 0:
    # CDFS_5886 (the original single-galaxy demo target) isn't in the current
    # fracAGN>0 sample - fall back to a "typical" galaxy (closest to the
    # population median Tier 1 residual for the best-performing template) so
    # this section still delivers a concrete visual sanity check.
    med_resid = catalog_cols[f'resid_median_{best_template}'].median()
    fallback_idx = (catalog_cols[f'resid_median_{best_template}'] - med_resid).abs().idxmin()
    spot_id = catalog_cols.loc[fallback_idx, 'ID']
    print(f"CDFS_5886 not in the fracAGN>0 sample - using {spot_id} "
          f"(typical {best_template} residual) as the spot-check galaxy instead.")
    spot_row = catalog_cols[catalog_cols['ID'] == spot_id]

if len(spot_row) == 0:
    print("No spot-check galaxy available.")
else:
    spot_row = spot_row.iloc[0]
    path = _fits_path(spot_id, spot_row['field'])
    full_sed = data_io.read_cigale_best_model(path, redshift=spot_row['redshift'], restframe=True)
    host_sed = analysis.decompose_cigale_sed(full_sed, target='host')

    fig, ax = plt.subplots(figsize=visualization.PASA_WIDE)
    ax.loglog(full_sed['lambda (Angstroms)'], full_sed['L_lambda_total'], color='k', label='Full (CIGALE)')
    ax.loglog(host_sed['lambda (Angstroms)'], host_sed['Total Flux (erg/s/cm^2/Angstrom)'], color='gray', ls='--',
               label='Host (decomposed)')
    for tname in TEMPLATES:
        composite = composite_math.create_composite_sed(TEMPLATES[tname], host_sed, spot_row['alpha_theory'])
        ax.loglog(composite['lambda (Angstroms)'], composite['Total Flux (erg/s/cm^2/Angstrom)'],
                   color=TEMPLATE_COLORS[tname], ls=':', label=f'Host + {tname} (theory)')
    ax.set_xlabel('Restframe wavelength (A)')
    ax.set_ylabel('Flux (L_lambda_total units)')
    ax.set_title(f"{spot_id} spot-check (fracAGN={spot_row['fracAGN']:.2f}, alpha_theory={spot_row['alpha_theory']:.3f})")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    for tname in TEMPLATES:
        print(f"{tname}: max_abs_resid={spot_row[f'max_abs_resid_{tname}']:.3f}, "
              f"dUV={spot_row[f'dUV_{tname}']:+.4f}, dVJ={spot_row[f'dVJ_{tname}']:+.4f}")

In [ ]:
print("=== Validation summary ===")
for tname in TEMPLATES:
    tier1_pass_frac = (summary_df[f'max_abs_resid_{tname}'] < 0.5).mean()
    tier2_med, tier2_lo, tier2_hi = bootstrap_ci(
        np.maximum(catalog_cols[f'dUV_{tname}'].abs(), catalog_cols[f'dVJ_{tname}'].abs()), np.median, seed=600)
    verdict = 'PASS' if tier2_hi < COLOUR_TOLERANCE else 'MARGINAL/FAIL'
    print(f"\n{tname}:")
    print(f"  Tier 1: {tier1_pass_frac:.1%} of galaxies have per-galaxy max relative residual < 50% "
          f"(wavelength > 1300 A, excluding the Lyman-continuum/IGM regime)")
    print(f"  Tier 2: median max(|dUV|,|dVJ|) = {tier2_med:.4f} [{tier2_lo:.4f},{tier2_hi:.4f}] "
          f"(tolerance {COLOUR_TOLERANCE} mag) -> {verdict}")
    print(f"  Tier 2 classification agreement: {overall_agreement[tname]:.1%}")

print(f"\nAlpha calibration ({best_template}): median alpha_fit/alpha_theory = {m_ratio:.2f} "
      f"[{lo_ratio:.2f},{hi_ratio:.2f}] (Spearman rho={rho:.3f})")
print(f"\nBest-performing template overall: {best_template}")

**Note:** no figures from this notebook are currently added to
`config.PAPER_FIGURE_MANIFEST` / the paper's `\appendix`
(`Context/AGNPaper/paper.tex:496-498`). If any of Figures 1-4 above should be
included as an appendix validation figure, add an entry to
`PAPER_FIGURE_MANIFEST` in `src/config.py` and a corresponding export cell
(see `Model_Validation_via_IRAC.ipynb`'s final cell for the pattern) - not
done here since that's a decision for the paper's author, not assumed by
this notebook.